# MongoDB e PostgreSQL

In [15]:
from pymongo import MongoClient

# Requires the PyMongo package.
# https://api.mongodb.com/python/current

client = MongoClient('mongodb+srv://<user>:<pw>@host/db')
result = client['ITS_NoSQL']['TWITCH'].aggregate([
    {
        '$group': {
            '_id': '$MOST_STREAMED_GAME', 
            'MediaViews': {
                '$avg': '$AVG_VIEWERS_PER_STREAM'
            }, 
            'TotalTime': {
                '$sum': '$TOTAL_TIME_STREAMED'
            }
        }
    }, {
        '$sort': {
            'MediaViews': -1
        }
    }
])
lista = []
for document in result:
    lista.append(document)

lista

[{'_id': 'NBA 2K22', 'MediaViews': 211701.0, 'TotalTime': 2882},
 {'_id': 'Special Events', 'MediaViews': 113156.0, 'TotalTime': 10493},
 {'_id': 'FIFA 21', 'MediaViews': 73795.33333333333, 'TotalTime': 14002},
 {'_id': 'Chess', 'MediaViews': 71906.0, 'TotalTime': 23115},
 {'_id': 'Destiny 2', 'MediaViews': 62148.857142857145, 'TotalTime': 78721},
 {'_id': 'Rocket League', 'MediaViews': 56851.57142857143, 'TotalTime': 28328},
 {'_id': 'Overwatch', 'MediaViews': 56038.42857142857, 'TotalTime': 158922},
 {'_id': 'F1 2019', 'MediaViews': 42554.0, 'TotalTime': 3960},
 {'_id': 'World of Warcraft', 'MediaViews': 40071.0, 'TotalTime': 214340},
 {'_id': 'League of Legends',
  'MediaViews': 37957.607142857145,
  'TotalTime': 716202},
 {'_id': 'Dungeons & Dragons', 'MediaViews': 37121.0, 'TotalTime': 2942},
 {'_id': 'Yu-Gi-Oh! Duel Links', 'MediaViews': 34434.0, 'TotalTime': 7875},
 {'_id': 'Fortnite', 'MediaViews': 33149.2, 'TotalTime': 220342},
 {'_id': 'Music', 'MediaViews': 32878.6, 'TotalTi

In [16]:
import pandas as pd

myDf = pd.DataFrame.from_records(lista)
myDf = myDf.rename(columns={'_id':'gametitle','MediaViews':'mediaviews','TotalTime':'totaltime'})
myDf

,gametitle,mediaviews,totaltime
0,NBA 2K22,211701.000000,2882
1,Special Events,113156.000000,10493
2,FIFA 21,73795.333333,14002
3,Chess,71906.000000,23115
4,Destiny 2,62148.857143,78721
...,...,...,...
101,Madden NFL 24,0.000000,3030
102,War Thunder,0.000000,2220
103,Honkai: Star Rail,0.000000,1727
104,Casino,0.000000,5530


## Importiamo su PostgreSQL

Supponendo di avere un database PostgreSQL a cui abbiamo accesso, altrimenti ce lo facciamo con Docker

In [17]:
from sqlalchemy import create_engine, Column, Integer, String, Float
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

# Definire il modello di base
Base = declarative_base()

# Definire il modello della tabella
class TwitchData(Base):
    __tablename__ = 'twitchdata'
    id = Column(Integer, primary_key=True, autoincrement=True)
    gametitle = Column(String, nullable=False)
    mediaviews = Column(Float, nullable=False)
    totaltime = Column(Integer, nullable=False)

# Crea il motore di connessione a PostgreSQL
engine = create_engine('postgresql+psycopg2://usr:pw@host/db')

# Crea la tabella (se non esiste)
Base.metadata.create_all(engine)

# Crea una sessione
Session = sessionmaker(bind=engine)
session = Session()


C:\Users\andre\AppData\Local\Temp\ipykernel_6880\3199640459.py:6: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [18]:
for index, row in myDf.iterrows():
    new_data = TwitchData(
        gametitle=row['gametitle'],
        mediaviews=row['mediaviews'],
        totaltime=row['totaltime']
    )
    session.add(new_data)

# Effettua il commit per salvare i dati
session.commit()

# Chiude la sessione
session.close()

Ora leggiamo i dati:

In [19]:
query = "SELECT * FROM twitchdata"

# Leggiamo i dati direttamente nel DataFrame Pandas
# engine è lo stesso creato prima con l'ORM
df = pd.read_sql(query, engine)
df

,id,gametitle,mediaviews,totaltime
0,1,NBA 2K22,211701.000000,2882
1,2,Special Events,113156.000000,10493
2,3,FIFA 21,73795.333333,14002
3,4,Chess,71906.000000,23115
4,5,Destiny 2,62148.857143,78721
...,...,...,...,...
313,314,Madden NFL 24,0.000000,3030
314,315,War Thunder,0.000000,2220
315,316,Honkai: Star Rail,0.000000,1727
316,317,Casino,0.000000,5530


## IRIS Dataset

In [20]:
url = 'https://gist.githubusercontent.com/netj/8836201/raw/6f9306ad21398ea43cba4f7d537619d0e07d5ae3/iris.csv'

irisData = pd.read_csv(url)
irisData

,sepal.length,sepal.width,petal.length,petal.width,variety
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa
3,4.6,3.1,1.5,0.2,Setosa
4,5.0,3.6,1.4,0.2,Setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,Virginica
146,6.3,2.5,5.0,1.9,Virginica
147,6.5,3.0,5.2,2.0,Virginica
148,6.2,3.4,5.4,2.3,Virginica


In [21]:
class IrisData(Base):
    __tablename__ = 'iris_data'
    id = Column(Integer, primary_key=True, autoincrement=True)
    sepal_length = Column('sepal_length', Float)
    sepal_width = Column('sepal_width', Float)
    petal_length = Column('petal_length', Float)
    petal_width = Column('petal_width', Float)
    variety = Column('variety', String)

# Rinomina le colonne del DataFrame per corrispondere ai nomi nel modello ORM
irisData.rename(columns={
    'sepal.length': 'sepal_length',
    'sepal.width': 'sepal_width',
    'petal.length': 'petal_length',
    'petal.width': 'petal_width'
}, inplace=True)

irisData

,sepal_length,sepal_width,petal_length,petal_width,variety
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa
3,4.6,3.1,1.5,0.2,Setosa
4,5.0,3.6,1.4,0.2,Setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,Virginica
146,6.3,2.5,5.0,1.9,Virginica
147,6.5,3.0,5.2,2.0,Virginica
148,6.2,3.4,5.4,2.3,Virginica


Creiamo la tabella sul database:

In [22]:
Base.metadata.create_all(engine)

inseriamo i dati

In [23]:
# Crea una sessione
Session = sessionmaker(bind=engine)
session = Session()

# Rinomina le colonne del DataFrame per corrispondere ai nomi nel modello ORM
irisData.rename(columns={
    'sepal.length': 'sepal_length',
    'sepal.width': 'sepal_width',
    'petal.length': 'petal_length',
    'petal.width': 'petal_width'
}, inplace=True)

# Itera su ogni riga del DataFrame e aggiungere i dati al database
for index, row in irisData.iterrows():
    iris = IrisData(
        sepal_length=row['sepal_length'],
        sepal_width=row['sepal_width'],
        petal_length=row['petal_length'],
        petal_width=row['petal_width'],
        variety=row['variety']
    )
    session.add(iris)

# Effettua il commit per salvare i dati
session.commit()

# Chiude la sessione
session.close()

print("Dati inseriti in PostgreSQL")

Dati inseriti in PostgreSQL
